# Exercice 1.4: Détection d'anomalie et validation

**Séance 5: Détection d'anomalie et validation**

---



## Contexte

Dans cet exercice, vous allez l'explorer le jeu de donnée avec différentes méthodes de détection d'anomalies pour décider quoi faire de chaque problème identifié.

---



## Objectifs d'apprentissage

- Distinguer les trois types d'anomalies sur des données réelles
- Implémenter et comparer quatre méthodes de détection d'outliers : IQR, modified z-score, Isolation Forest, LOF
- Comprendre quand chaque méthode est appropriée
- Poser les premières briques d'une validation systématique avec Great Expectations
- Porter un jugement métier : toutes les anomalies ne sont pas des erreurs

---



## Section 1: Comparaison de 4 méthodes de détection d'outliers

**Objectif** : appliquer IQR, modified z-score, Isolation Forest et DBSCAN sur deux variables pertinentes pour votre hypothèse, et comparer leurs résultats de façon systématique.



### 1.1 Préparation

**Ce que vous devez faire :**
- Seléctionnez deux variables pertientes pour votre hypothèses. Préparez un DataFrame de travail sur ces deux colonnes en enlevant les valeurs manquantes s'il y en a.
- Refactorisez vos implémentations de l'Exercice 1.1 en **fonctions réutilisables** avec des docstrings.



### 1.2 Application des méthodes statistiques
- Appliquez vos deux fonctions sur vos colonnes séparément. Stockez les résultats dans des colonnes de votre DataFrame de travail (`outlier_iqr_price`, `outlier_iqr_area`, etc.).



### 1.3 Isolation Forest
- Appliquez Isolation Forest sur ces mêmes colonnes **simultanément**.

> `IsolationForest` de `sklearn.ensemble` s'entraîne avec `.fit_predict()`: il retourne `1` (normal) et `-1` (outlier). Fixez un random state et utilisez `contamination='auto'`.
>
> L'intérêt d'appliquer Isolation Forest sur les deux variables ensemble : il peut détecter des **outliers contextuels**.



### 1.4 DBSCAN
- Normalisez vos données pour ramenez vos deux colonnes sur une échèle similaire si besoin
- Appliquez DBSCAN sur les mêmes deux variables.

> Après normalisation, commencez avec `eps=0.5, min_samples=10` et ajustez si vous obtenez trop ou trop peu d'outliers. Les points avec le label `-1` sont les anomalies.

### 1.5 Visualisation et comparaison

**Ce que vous devez produire :**
- Un **scatterplot** de vos colonnes coloré par catégorie de statut : normal / détecté par 1 méthode seulement / détecté par 2-3 méthodes / détecté par toutes les méthodes.

> Construisez une colonne de statut avec `np.select()` en combinant vos masques booléens. Comptez le nombre de méthodes qui signalent chaque point comme outlier: c'est plus informatif que binaire.

- Le **tableau comparatif** suivant, complété avec vos résultats :

| Méthode | Nb outliers SalePrice | % total | Nb outliers GrLivArea | % total | Nb outliers (les deux variables) |
|---------|----------------------|---------|-----------------------|---------|----------------------------------|
| IQR | | | | | |
| Modified Z-score | | | | | |
| Isolation Forest |: |: |: |: | |
| DBSCAN |: |: |: |: | |
| Intersection (toutes méthodes) | | | | | |

> Isolation Forest et DBSCAN opèrent sur les deux variables ensemble: leur résultat est un masque unique, pas deux masques séparés. Les cellules "—" reflètent cette différence conceptuelle.



**Dans une cellule Markdown, répondez :**
- Quelle méthode est la plus conservative ? La plus agressive ?
- Identifiez une observation (donnez son index) détectée par Isolation Forest ou DBSCAN mais **pas** par IQR ni modified z-score. Affichez ses valeurs et expliquez pourquoi elle est anormale dans la combinaison des deux variables mais pas individuellement.
- Pour prédire `SalePrice` (ou la colonne pertinente pour votre hypothèse) avec une régression linéaire, quelle méthode de détection vous semble la plus adaptée comme pré-traitement ? Justifiez.

---
## Section 2: LOF et anomalies contextuelles

**Objectif** : exploiter les scores continus du LOF pour nuancer la détection binaire.

**Ce que vous devez faire :**

- Appliquez LOF sur vos variables. Récupérez les prédictions binaires ET les **scores d'anomalie continus**.

> Après `fit_predict()`, l'attribut `negative_outlier_factor_` donne un score par point. Proche de -1 = normal, très négatif (ex. -5, -10) = fortement anormal. Ce score existe uniquement après l'appel à `fit_predict()`.

- Créez un scatterplot coloré par le **score LOF continu**.

> Utilisez `hue=scores_lof` avec `palette='viridis_r'` (inversé : les valeurs les plus anormales ressortent dans les tons chauds). Une colorbar ou une légende graduée rend le graphique plus lisible que des points tous de même taille.

- Affichez les **5 observations avec le score LOF le plus extrême**

> `pd.Series(scores_lof).nsmallest(5)` donne les indices des 5 scores les plus bas (les plus anormaux). Utilisez ces indices pour récupérer les lignes dans le DataFrame original.


**Dans une cellule Markdown, répondez :**
- Ces 5 observations vous semblent-elles des erreurs ou des maisons réellement atypiques ? Quels indices dans les autres colonnes vous permettent de pencher dans un sens ou dans l'autre ?
- Testez `n_neighbors=5` puis `n_neighbors=50`. Les 5 outliers les plus extrêmes changent-ils ? Qu'est-ce que cela implique sur la robustesse de LOF ?
- Expliquez avec vos propres mots pourquoi un point peut avoir des valeurs individuellement normales mais être une anomalie contextuelle.


## Section 3: Stratégies de traitement

**Objectif** : décider quoi faire de chaque anomalie identifiée.

**Ce que vous devez faire :**
- Sélectionnez **5 observations** parmi les outliers détectés: choisissez des cas variés : certains clairement aberrants, d'autres ambigus. Complétez ce tableau :

| Index | Variable(s) anormale(s) | Valeur(s) | Type d'anomalie | Stratégie choisie | Justification |
|-------|------------------------|-----------|-----------------|-------------------|---------------|
| ... | ... | ... | Ponctuelle / Contextuelle | Suppression / Winsorisation / Log-transform / Conservation | ... |
- Appliquez la **log-transformation** sur `SalePrice` et `LotArea`. Comparez les distributions avant et après avec des histogrammes côte à côte incluant le skewness.

> `np.log1p()` est préférable à `np.log()` car elle gère les valeurs à zéro. Calculez `df['SalePrice'].skew()` avant et après transformation: le skewness devrait se rapprocher de 0.



**Dans une cellule Markdown, répondez :**
- De combien le skewness a-t-il diminué après log-transformation pour chaque variable ?
- La log-transformation a-t-elle rendu la détection d'outliers par z-score plus ou moins conservative ? Relancez `detect_outliers_modified_zscore()` sur les valeurs transformées et comparez.
- Dans quelle situation préféreriez-vous la winsorisation à la suppression pure ?

--


## Section 4: Validation avec Great Expectations

**Objectif** : formaliser votre connaissance du dataset en règles de validation reproductibles.

>  Vous utiliserez le mode simple de Great Expectations (`ge.from_pandas()`) sans configurer un contexte complet.

**Ce que vous devez faire :**

- Créez un DataFrame Great Expectations à partir du dataset nettoyé (Section 3).

> `import great_expectations as ge` puis `df_ge = ge.from_pandas(df_clean)`. Cela n'est pas destructif: cela ajoute des méthodes `expect_*` sans modifier les données.

- Définissez **6 règles de validation** basées sur votre EDA. Chaque règle doit être justifiée :

| Règle GE | Variable | Valeur(s) | Justification (ce que vous avez observé) |
|----------|----------|-----------|------------------------------------------|
| `expect_column_values_to_not_be_null` | `SalePrice` |: | Prix de vente toujours renseigné dans vos données |
| `expect_column_values_to_be_between` | `OverallQual` | 1 à 10 | Variable ordinale bornée par définition |
| ... | ... | ... | ... |

> Autres méthodes utiles : `expect_column_values_to_be_in_set` (pour les variables catégorielles avec un ensemble fini de valeurs), `expect_column_mean_to_be_between` (pour détecter une dérive de moyenne), `expect_table_columns_to_match_set` (pour vérifier que toutes les colonnes attendues sont présentes).

- Exécutez `df_ge.validate()` et analysez les résultats.



**Dans une cellule Markdown, répondez :**
- Quelles règles ont échoué ? Était-ce prévisible au vu de votre EDA ?
- Quelle différence entre `expect_column_values_to_not_be_null` et un simple `assert df['col'].notnull().all()` ? Dans quel contexte Great Expectations apporte-t-il une valeur réelle ?
- Si ces validations tournaient automatiquement à chaque livraison mensuelle de nouvelles données, quelle règle serait la plus utile pour détecter une dérive du dataset ?


## Section 5: Bilan critique : erreur ou événement légitime ?

**Ce que vous devez faire :**

24. Classez les anomalies les plus marquantes de l'exercice dans ce tableau :

| Observation ou groupe | Anomalie | Conclusion | Argument principal |
|----------------------|----------|------------|-------------------|
| ... | ... | Erreur / Événement légitime / Incertain | ... |

25. Choisissez **un cas incertain** et rédigez un paragraphe structuré :
   - Description de l'observation
   - Arguments en faveur d'une **erreur de données**
   - Arguments en faveur d'un **événement réel**
   - Quelle information supplémentaire vous manque pour trancher ?
   - Dans une équipe professionnelle, à qui poseriez-vous la question ?

> La réponse à "est-ce une erreur ou un événement réel ?" vient rarement des données seules: elle vient des experts métier. Ici, un expert immobilier d'Ames Iowa 2006-2010 serait la bonne personne à consulter.

---



## Checklist avant de rendre

- [ ] Le notebook s'exécute du début à la fin sans erreur (`Restart & Run All`)
- [ ] Les fonctions `detect_outliers_iqr()` et `detect_outliers_modified_zscore()` ont des docstrings et sont appelées (pas dupliquées) dans les sections suivantes
- [ ] Le tableau comparatif Section 4.5 est complet avec des valeurs numériques réelles
- [ ] Le scatterplot Section 4.5 distingue au moins 4 catégories de statut
- [ ] Les scores LOF continus sont visualisés avec une palette graduelle (pas binaire)
- [ ] Le tableau de 6 règles Great Expectations est complété avec les justifications EDA
- [ ] La Section 8 contient le tableau de classification ET le paragraphe structuré sur le cas incertain